In [1]:
import pandas as pd
import os
import numpy as np
import math
import pytaxonkit
import subprocess
import json

In [50]:
!datasets download virus genome accession "AF538689.1"  --include protein --filename "test-data.zip"

Downloading: test-data.zip    1.52kB 9.18MB/s
Downloading: test-data.zip    1.52kB 9.18MB/s
Downloading: test-data.zip    4.07kB valid data package
Validating package files [================================================] 100% 5/5


In [137]:
!datasets summary taxonomy taxon "Barley"

Error: The taxonomy name 'Barley' is not exact. Try using one of the suggested taxids:
Barley yellow dwarf virus (species, taxid: 12037, viruses)
Barley stripe mosaic virus (no-rank, taxid: 12327, viruses)
Barley yellow dwarf virus-PAV (no-rank, taxid: 2169986, viruses)
Barley dwarf virus (no-rank, taxid: 497862, viruses)
Barley yellow dwarf virus MAV (no-rank, taxid: 2169984, viruses)
Barley virus G (no-rank, taxid: 1825924, viruses)
Cytorhabdovirus hordei (no-rank, taxid: 1985699, Barley yellow striate mosaic virus)
Barley yellow dwarf virus-PAS (no-rank, taxid: 2169985, viruses)
Barley yellow mosaic virus (no-rank, taxid: 12465, viruses)
Barley mild mosaic virus (no-rank, taxid: 12466, viruses)

Use datasets summary taxonomy taxon <command> --help for detailed help about a command.



In [2]:
dataset_csv_filepath = os.path.join(os.getcwd(), "..", "..", "..","..", "..", "input/data/bvbrc/BVBRC_reference_genome_20260429.csv")
df = pd.read_csv(dataset_csv_filepath, low_memory=False)
print(f"Dataset size: {df.shape}")
df.columns

Dataset size: (15731, 90)


Index(['Genome ID', 'Genome Name', 'Other Names', 'NCBI Taxon ID',
       'Taxon Lineage IDs', 'Taxon Lineage Names', 'Superkingdom', 'Kingdom',
       'Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species',
       'Genome Status', 'Strain', 'Serovar', 'Biovar', 'Pathovar', 'MLST',
       'Segment', 'Subtype', 'H_type', 'N_type', 'H1 Clade Global',
       'H1 Clade US', 'H5 Clade', 'pH1N1-like', 'Lineage', 'Clade', 'Subclade',
       'Other Typing', 'Culture Collection', 'Type Strain', 'Reference',
       'Genome Quality', 'Completion Date', 'Publication', 'Authors',
       'BioProject Accession', 'BioSample Accession', 'Assembly Accession',
       'SRA Accession', 'GenBank Accessions', 'Sequencing Center',
       'Sequencing Status', 'Sequencing Platform', 'Sequencing Depth',
       'Assembly Method', 'Chromosome', 'Plasmids', 'Contigs', 'Size',
       'GC Content', 'Contig L50', 'Contig N50', 'TRNA', 'RRNA', 'Mat Peptide',
       'CDS', 'Coarse Consistency', 'Fine Consistency', 'Ch

In [3]:
df

,Genome ID,Genome Name,Other Names,NCBI Taxon ID,Taxon Lineage IDs,Taxon Lineage Names,Superkingdom,Kingdom,Phylum,Class,...,Host Age,Host Health,Host Group,Lab Host,Passage,Other Clinical,Additional Metadata,Comments,Date Inserted,Date Modified
0,1000373.10,Rosellinia necatrix quadrivirus 1 W1118,NaN,1000373,10239;2559587;2732396;2732405;2732458;2732540;...,Viruses;Riboviria;Orthornavirae;Duplornavirico...,Viruses,Orthornavirae,Duplornaviricota,Chrymotiviricetes,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-12-06T11:05:04.123Z,2025-12-06T11:05:04.123Z
1,1000373.40,Rosellinia necatrix quadrivirus 1 W1075,NaN,1000373,10239;2559587;2732396;2732405;2732458;2732540;...,Viruses;Riboviria;Orthornavirae;Duplornavirico...,Viruses,Orthornavirae,Duplornaviricota,Chrymotiviricetes,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-12-06T08:01:47.037Z,2025-12-06T08:01:47.037Z
2,1000373.50,Rosellinia necatrix quadrivirus 1 W1075,NaN,1000373,10239;2559587;2732396;2732405;2732458;2732540;...,Viruses;Riboviria;Orthornavirae;Duplornavirico...,Viruses,Orthornavirae,Duplornaviricota,Chrymotiviricetes,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-12-06T08:01:47.164Z,2025-12-06T08:01:47.164Z
3,1000411.30,Jacunda virus,NaN,1000411,10239;2559587;2732396;2497569;2497571;3151693;...,Viruses;Riboviria;Orthornavirae;Negarnaviricot...,Viruses,Orthornavirae,Negarnaviricota,Bunyaviricetes,...,NaN,NaN,Nonhuman Mammal,NaN,NaN,NaN,NaN,NaN,2021-01-23T22:04:18.329Z,2021-01-23T22:04:18.329Z
4,1000411.40,Jacunda virus,NaN,1000411,10239;2559587;2732396;2497569;2497571;3151693;...,Viruses;Riboviria;Orthornavirae;Negarnaviricot...,Viruses,Orthornavirae,Negarnaviricota,Bunyaviricetes,...,NaN,NaN,Nonhuman Mammal,NaN,NaN,NaN,NaN,NaN,2021-01-23T22:06:18.856Z,2021-01-23T22:06:18.856Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15726,999729.23,Leanyer virus AusN16701,NaN,999729,10239;2559587;2732396;2497569;2497571;3151693;...,Viruses;Riboviria;Orthornavirae;Negarnaviricot...,Viruses,Orthornavirae,Negarnaviricota,Bunyaviricetes,...,NaN,NaN,Insect,NaN,NaN,NaN,NaN,NaN,2022-03-12T21:40:21.986Z,2022-03-12T21:40:21.986Z
15727,999729.24,Leanyer virus AusN16701,NaN,999729,10239;2559587;2732396;2497569;2497571;3151693;...,Viruses;Riboviria;Orthornavirae;Negarnaviricot...,Viruses,Orthornavirae,Negarnaviricota,Bunyaviricetes,...,NaN,NaN,Insect,NaN,NaN,NaN,NaN,NaN,2022-03-12T21:34:20.054Z,2022-03-12T21:34:20.054Z
15728,999729.25,Leanyer virus AusN16701,NaN,999729,10239;2559587;2732396;2497569;2497571;3151693;...,Viruses;Riboviria;Orthornavirae;Negarnaviricot...,Viruses,Orthornavirae,Negarnaviricota,Bunyaviricetes,...,NaN,NaN,Insect,NaN,NaN,NaN,NaN,NaN,2022-03-12T21:49:25.104Z,2022-03-12T21:49:25.104Z
15729,999729.35,Leanyer virus AusN16701,NaN,999729,10239;2559587;2732396;2497569;2497571;3151693;...,Viruses;Riboviria;Orthornavirae;Negarnaviricot...,Viruses,Orthornavirae,Negarnaviricota,Bunyaviricetes,...,NaN,NaN,Insect,NaN,NaN,NaN,NaN,NaN,2022-05-01T01:51:20.496Z,2022-05-01T01:51:20.496Z


In [4]:
df["Host Group"].value_counts()

Nonhuman Mammal    1395
Bacteria            882
Plant               779
Insect              770
Human               610
Avian               371
Tick                298
Lab                  79
Fish                 71
Reptile              35
Sea Mammal           25
Crustaceans          20
Amphibian             2
Fungi                 1
Name: Host Group, dtype: int64

In [5]:
df["Host Name"].nunique()

3771

In [26]:
df[df["Host Name"].isna()][["Genome Name", "GenBank Accessions"]]

,Genome Name,GenBank Accessions
33,Ljungan virus M1146,AF538689
35,Bos taurus papillomavirus 7,DQ217793
45,Simian sapelovirus 1 2383,NC_004451
46,Simian sapelovirus 1 2383,AY064708
47,Porcine sapelovirus 1 V13,AF406813
...,...,...
15631,Plutella xylostella granulovirus K1,AF270937
15633,Pseudomonas phage phi15,FR823298
15634,Tanapox virus TPV-Kenya,EF420156
15672,Titi monkey adenovirus ECC-2011,HQ913600


In [7]:
print(df["Host Name"].nunique())
print(df["Host Common Name"].nunique())

3771
369


In [10]:
print(sum(df["Host Name"].isna()))
print(sum(df["Host Common Name"].isna()))
print(sum(df["Host Name"].isna() & ~df["Host Common Name"].isna() ))

4349
10393
34


In [6]:
df[df["Host Name"].isna() & ~df["Host Common Name"].isna()][["Genome ID", "GenBank Accessions", "Host Name", "Host Common Name"]]

,Genome ID,GenBank Accessions,Host Name,Host Common Name
125,1.024541e+04,NC_006998,NaN,Lab host
126,1.025430e+04,AY243312,NaN,Lab host
519,1.078429e+04,NC_001540,NaN,Lab host
520,1.078829e+04,NC_001539,NaN,Lab host
521,1.079439e+04,NC_001510,NaN,Lab host
522,1.079440e+04,J02275,NaN,Lab host
951,1.133292e+06,JN939331,NaN,Lab host
952,1.133294e+06,JN939332,NaN,Lab host
1189,1.168745e+06,JQ797329,NaN,Lab host
2082,1.327970e+06,KC821606,NaN,Lab host


In [15]:
sum(~df["Host Name"].isna())

11382

In [7]:
df = df[~df["Host Name"].isna()]
df.shape

(11382, 90)

In [88]:
!datasets summary taxonomy taxon "Homo sapiens, Sus scrofa"

{"reports": [{"query":["Homo sapiens"],"taxonomy":{"children":[741158,63221],"classification":{"class":{"id":40674,"name":"Mammalia"},"domain":{"id":2759,"name":"Eukaryota"},"family":{"id":9604,"name":"Hominidae"},"genus":{"id":9605,"name":"Homo"},"kingdom":{"id":33208,"name":"Metazoa"},"order":{"id":9443,"name":"Primates"},"phylum":{"id":7711,"name":"Chordata"},"species":{"id":9606,"name":"Homo sapiens"}},"counts":[{"count":2497,"type":"COUNT_TYPE_ASSEMBLY"},{"count":193866,"type":"COUNT_TYPE_GENE"},{"count":701,"type":"COUNT_TYPE_tRNA"},{"count":785,"type":"COUNT_TYPE_rRNA"},{"count":168,"type":"COUNT_TYPE_snRNA"},{"count":4,"type":"COUNT_TYPE_scRNA"},{"count":1201,"type":"COUNT_TYPE_snoRNA"},{"count":20624,"type":"COUNT_TYPE_PROTEIN_CODING"},{"count":22583,"type":"COUNT_TYPE_ncRNA"},{"count":128261,"type":"COUNT_TYPE_BIOLOGICAL_REGION"},{"count":843,"type":"COUNT_TYPE_OTHER"}],"curator_common_name":"human","current_scientific_name":{"authority":"Linnaeus, 1758","name":"Homo sapiens"

In [140]:
VERTEBRATA_TAX_ID = 7742

def get_host_metadata(host_names):
    host_metadata = {}
    error_hosts = []
    n = len(host_names)
    for i, host_name in enumerate(host_names):
        print(f"({i}) {host_name}")
        try: 
            output = subprocess.run(["datasets", "summary", "taxonomy", "taxon", host_name], capture_output=True)
            #tax_json = json.loads(output.stdout)["reports"][0]["taxonomy"]["classification"]
            output_json = json.loads(output.stdout)
            reports = output_json["reports"]
            report_count = int(output_json["total_count"])
            for report in reports:
                tax_json = report["taxonomy"]
                classification_json = tax_json["classification"]
                metadata = {}
                if "species" in classification_json:
                    metadata["species_tax_name"] = classification_json["species"]["name"]
                    metadata["species_tax_id"] = classification_json["species"]["id"]
    
                if "genus" in classification_json:
                    metadata["genus_tax_name"] = classification_json["genus"]["name"]
                    metadata["genus_tax_id"] = classification_json["genus"]["id"]
    
                if "family" in tax_json:
                    metadata["family_tax_name"] = classification_json["family"]["name"]
                    metadata["family_tax_id"] = classification_json["family"]["id"]
    
                parents = tax_json["parents"]
                if VERTEBRATA_TAX_ID in parents:
                    metadata["is_vertebrate"] = True
                else:
                    metadata["is_vertebrate"] = False
                    
                host_metadata[host_name] = metadata
        except Exception as e:
            error_hosts.append(host_name)
            print(e)
    return host_metadata, error_hosts

In [78]:
df["host_name_cleaned"] = df["Host Name"].apply(lambda x: " ".join(x.split(" ")[:2]))

/tmp/ipykernel_1450585/3428461402.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["host_name_cleaned"] = df["Host Name"].apply(lambda x: " ".join(x.split(" ")[:2]))


In [79]:
df["Host Name"].nunique()

3771

In [80]:
df["Host Name"].value_counts()

Homo sapiens                           577
Mycobacterium smegmatis mc2 155        172
Escherichia coli                       131
Klebsiella pneumoniae                  118
Pseudomonas aeruginosa                  85
                                      ... 
Salmo salar l. (atlantic salmon)         1
Cicer arietinum voucher dar 76857        1
Cicer arietinum voucher brip 52877a      1
Cicer arietinum voucher brip 52878a      1
Digitaria didactyla willd                1
Name: Host Name, Length: 3771, dtype: int64

In [142]:
df["host_name_cleaned"].nunique()

2635

In [143]:
df["host_name_cleaned"].value_counts()

Homo sapiens               579
Escherichia coli           302
Mycobacterium smegmatis    183
Klebsiella pneumoniae      174
Pseudomonas aeruginosa     143
                          ... 
Feral pigeon                 1
Cercopithecus mitis          1
Datura inoxia                1
Ictalurus punctatus          1
Rhododendron maximum         1
Name: host_name_cleaned, Length: 2635, dtype: int64

In [144]:
hosts = df["host_name_cleaned"].unique()

In [106]:
for i, host in enumerate(hosts):
    if host == "Lycopersicon esculentum":
        break

print(i)
hosts[143]

143


'Lycopersicon esculentum'

In [145]:
host_metadata, error_hosts = get_host_metadata(hosts)

(0) Rosellinia necatrix
(1) Rodent
Expecting value: line 1 column 1 (char 0)
(2) Sandfly
Expecting value: line 1 column 1 (char 0)
(3) Homo sapiens
(4) mosquito
Expecting value: line 1 column 1 (char 0)
(5) Chilli
Expecting value: line 1 column 1 (char 0)
(6) Scalopus aquaticus
(7) Vernonia cinerea
(8) Perca fluviatilis
(9) Limnodynastes ornatus
(10) Gallus gallus
(11) Drosophila ananassae
(12) Diptera
(13) Sida sp.
(14) Shigella flexneri
(15) Hardenbergia comptoniana
(16) Spissistilus festinus
(17) Lutzomyia sp.
(18) Staphylococcus aureus
(19) Ipomoea batatas
(20) Ipomoea purpurea
(21) Ipomoea indica
(22) Rhodococcus equi
(23) Trichomonas vaginalis
(24) Triticum aestivum
(25) Ixodes (Trichotoixodes)
Expecting value: line 1 column 1 (char 0)
(26) Uria sp.
Expecting value: line 1 column 1 (char 0)
(27) Musa sp.
(28) Euphorbia heterophylla
(29) cutthroat trout
(30) Thysanoplusia orichalcea
(31) raccoon
(32) Columba livia
(33) sheep
(34) Solanum lycopersicum
(35) Stenotrophomonas maltophi

In [91]:
len(host_metadata)

2187

In [94]:
df["virus_host_species_tax_id"] = df["host_name_cleaned"].apply(lambda x: host_metadata.get(x).get("species_tax_id") if x in host_metadata else None)

/tmp/ipykernel_1450585/3630995619.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["virus_host_species_tax_id"] = df["host_name_cleaned"].apply(lambda x: host_metadata.get(x).get("species_tax_id") if x in host_metadata else None)


In [95]:
df["virus_host_is_vertebrate"] = df["host_name_cleaned"].apply(lambda x: host_metadata.get(x).get("is_vertebrate") if x in host_metadata else None)

/tmp/ipykernel_1450585/3631739729.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["virus_host_is_vertebrate"] = df["host_name_cleaned"].apply(lambda x: host_metadata.get(x).get("is_vertebrate") if x in host_metadata else None)


In [96]:
df 

,Genome ID,Genome Name,Other Names,NCBI Taxon ID,Taxon Lineage IDs,Taxon Lineage Names,Superkingdom,Kingdom,Phylum,Class,...,Lab Host,Passage,Other Clinical,Additional Metadata,Comments,Date Inserted,Date Modified,host_name_cleaned,virus_host_species_tax_id,virus_host_is_vertebrate
0,1000373.10,Rosellinia necatrix quadrivirus 1 W1118,NaN,1000373,10239;2559587;2732396;2732405;2732458;2732540;...,Viruses;Riboviria;Orthornavirae;Duplornavirico...,Viruses,Orthornavirae,Duplornaviricota,Chrymotiviricetes,...,NaN,NaN,NaN,NaN,NaN,2025-12-06T11:05:04.123Z,2025-12-06T11:05:04.123Z,Rosellinia necatrix,77044.0,False
1,1000373.40,Rosellinia necatrix quadrivirus 1 W1075,NaN,1000373,10239;2559587;2732396;2732405;2732458;2732540;...,Viruses;Riboviria;Orthornavirae;Duplornavirico...,Viruses,Orthornavirae,Duplornaviricota,Chrymotiviricetes,...,NaN,NaN,NaN,NaN,NaN,2025-12-06T08:01:47.037Z,2025-12-06T08:01:47.037Z,Rosellinia necatrix,77044.0,False
2,1000373.50,Rosellinia necatrix quadrivirus 1 W1075,NaN,1000373,10239;2559587;2732396;2732405;2732458;2732540;...,Viruses;Riboviria;Orthornavirae;Duplornavirico...,Viruses,Orthornavirae,Duplornaviricota,Chrymotiviricetes,...,NaN,NaN,NaN,NaN,NaN,2025-12-06T08:01:47.164Z,2025-12-06T08:01:47.164Z,Rosellinia necatrix,77044.0,False
3,1000411.30,Jacunda virus,NaN,1000411,10239;2559587;2732396;2497569;2497571;3151693;...,Viruses;Riboviria;Orthornavirae;Negarnaviricot...,Viruses,Orthornavirae,Negarnaviricota,Bunyaviricetes,...,NaN,NaN,NaN,NaN,NaN,2021-01-23T22:04:18.329Z,2021-01-23T22:04:18.329Z,Rodent,NaN,None
4,1000411.40,Jacunda virus,NaN,1000411,10239;2559587;2732396;2497569;2497571;3151693;...,Viruses;Riboviria;Orthornavirae;Negarnaviricot...,Viruses,Orthornavirae,Negarnaviricota,Bunyaviricetes,...,NaN,NaN,NaN,NaN,NaN,2021-01-23T22:06:18.856Z,2021-01-23T22:06:18.856Z,Rodent,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15726,999729.23,Leanyer virus AusN16701,NaN,999729,10239;2559587;2732396;2497569;2497571;3151693;...,Viruses;Riboviria;Orthornavirae;Negarnaviricot...,Viruses,Orthornavirae,Negarnaviricota,Bunyaviricetes,...,NaN,NaN,NaN,NaN,NaN,2022-03-12T21:40:21.986Z,2022-03-12T21:40:21.986Z,Anopheles meraukensis,59157.0,False
15727,999729.24,Leanyer virus AusN16701,NaN,999729,10239;2559587;2732396;2497569;2497571;3151693;...,Viruses;Riboviria;Orthornavirae;Negarnaviricot...,Viruses,Orthornavirae,Negarnaviricota,Bunyaviricetes,...,NaN,NaN,NaN,NaN,NaN,2022-03-12T21:34:20.054Z,2022-03-12T21:34:20.054Z,Anopheles meraukensis,59157.0,False
15728,999729.25,Leanyer virus AusN16701,NaN,999729,10239;2559587;2732396;2497569;2497571;3151693;...,Viruses;Riboviria;Orthornavirae;Negarnaviricot...,Viruses,Orthornavirae,Negarnaviricota,Bunyaviricetes,...,NaN,NaN,NaN,NaN,NaN,2022-03-12T21:49:25.104Z,2022-03-12T21:49:25.104Z,Anopheles meraukensis,59157.0,False
15729,999729.35,Leanyer virus AusN16701,NaN,999729,10239;2559587;2732396;2497569;2497571;3151693;...,Viruses;Riboviria;Orthornavirae;Negarnaviricot...,Viruses,Orthornavirae,Negarnaviricota,Bunyaviricetes,...,NaN,NaN,NaN,NaN,NaN,2022-05-01T01:51:20.496Z,2022-05-01T01:51:20.496Z,Anopheles meraukensis,59157.0,False


In [108]:
host_metadata["Lycopersicon esculentum"]

KeyError: 'Lycopersicon esculentum'

In [98]:
df.shape

(11382, 93)

In [99]:
df[df["virus_host_species_tax_id"].isna()]

,Genome ID,Genome Name,Other Names,NCBI Taxon ID,Taxon Lineage IDs,Taxon Lineage Names,Superkingdom,Kingdom,Phylum,Class,...,Lab Host,Passage,Other Clinical,Additional Metadata,Comments,Date Inserted,Date Modified,host_name_cleaned,virus_host_species_tax_id,virus_host_is_vertebrate
3,1000411.3,Jacunda virus,NaN,1000411,10239;2559587;2732396;2497569;2497571;3151693;...,Viruses;Riboviria;Orthornavirae;Negarnaviricot...,Viruses,Orthornavirae,Negarnaviricota,Bunyaviricetes,...,NaN,NaN,NaN,NaN,NaN,2021-01-23T22:04:18.329Z,2021-01-23T22:04:18.329Z,Rodent,NaN,None
4,1000411.4,Jacunda virus,NaN,1000411,10239;2559587;2732396;2497569;2497571;3151693;...,Viruses;Riboviria;Orthornavirae;Negarnaviricot...,Viruses,Orthornavirae,Negarnaviricota,Bunyaviricetes,...,NaN,NaN,NaN,NaN,NaN,2021-01-23T22:06:18.856Z,2021-01-23T22:06:18.856Z,Rodent,NaN,None
5,1000411.5,Jacunda virus,NaN,1000411,10239;2559587;2732396;2497569;2497571;3151693;...,Viruses;Riboviria;Orthornavirae;Negarnaviricot...,Viruses,Orthornavirae,Negarnaviricota,Bunyaviricetes,...,NaN,NaN,NaN,NaN,NaN,2021-01-23T22:06:19.344Z,2021-01-23T22:06:19.344Z,Rodent,NaN,None
6,1000645.3,Ariquemes virus,NaN,1000645,10239;2559587;2732396;2497569;2497571;3151693;...,Viruses;Riboviria;Orthornavirae;Negarnaviricot...,Viruses,Orthornavirae,Negarnaviricota,Bunyaviricetes,...,NaN,NaN,NaN,NaN,NaN,2021-01-23T21:53:37.675Z,2021-01-23T21:53:37.675Z,Sandfly,NaN,None
7,1000645.4,Ariquemes virus,NaN,1000645,10239;2559587;2732396;2497569;2497571;3151693;...,Viruses;Riboviria;Orthornavirae;Negarnaviricot...,Viruses,Orthornavirae,Negarnaviricota,Bunyaviricetes,...,NaN,NaN,NaN,NaN,NaN,2021-01-23T21:53:37.850Z,2021-01-23T21:53:37.850Z,Sandfly,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15535,935473.5,Rhynchosia yellow mosaic India virus Thiruvana...,NaN,935473,10239;2731342;2732092;2732416;2732424;2732539;...,Viruses;Monodnaviria;Shotokuvirae;Cressdnaviri...,Viruses,Shotokuvirae,Cressdnaviricota,Repensiviricetes,...,NaN,NaN,NaN,NaN,NaN,2025-12-16T15:07:07.341Z,2025-12-16T15:07:07.341Z,Rhynchosia,NaN,False
15536,935473.7,Rhynchosia yellow mosaic India virus Thiruvana...,NaN,935473,10239;2731342;2732092;2732416;2732424;2732539;...,Viruses;Monodnaviria;Shotokuvirae;Cressdnaviri...,Viruses,Shotokuvirae,Cressdnaviricota,Repensiviricetes,...,NaN,NaN,NaN,NaN,NaN,2025-12-16T15:07:11.283Z,2025-12-16T15:07:11.283Z,Rhynchosia,NaN,False
15580,943272.4,Honeysuckle ringspot virus California,NaN,943272,10239;2559587;2732396;2732406;2732463;2788833;...,Viruses;Riboviria;Orthornavirae;Kitrinoviricot...,Viruses,Orthornavirae,Kitrinoviricota,Tolucaviricetes,...,NaN,NaN,NaN,NaN,NaN,2025-12-18T00:03:53.154Z,2025-12-18T00:03:53.154Z,Honeysuckle,NaN,None
15715,996311.4,Tomato leaf curl Madagascar virus-Menabe [Mada...,NaN,996311,10239;2731342;2732092;2732416;2732424;2732539;...,Viruses;Monodnaviria;Shotokuvirae;Cressdnaviri...,Viruses,Shotokuvirae,Cressdnaviricota,Repensiviricetes,...,NaN,NaN,NaN,NaN,NaN,2025-12-07T16:05:56.968Z,2025-12-07T16:05:56.968Z,Lycopersicon esculentum,NaN,None


In [107]:
set(df["host_name_cleaned"].unique()) - set(host_metadata.keys())

{'1802)',
 'Abelmoscus esculentus',
 'Abelmoshus esculentus',
 'Abutilon sp.',
 'Acalypha sp.',
 'Adult cotesia',
 'Adult female',
 'Aedeomyia (ochler)',
 'Aeonium sp.',
 'Aeromonas hydrophila;Aeromonas',
 'African green',
 'African penguin',
 'Agapanthus sp.',
 'Aiptasia sp.',
 'Albelmoschus esculentus',
 'Alcea sp.',
 'Alpinia sp.',
 'Althea rosea',
 'Amblyomma (Theileriella)',
 'American white-tailed',
 'Anatid species',
 'Anthurium sp.',
 'Apple tree',
 'Arachis hypogea',
 'Arachis hypogeae',
 'Asgard group',
 'Ash gourd',
 'Ashy-headed sheldgoose',
 'Asian lion',
 'Asteraceae)',
 'Atcc 33303',
 'Bacilluls thuringiensis',
 'Bacillus cereus/thuringiensis',
 'Bacillus pseudalcaliphilus',
 'Bacillus sp.',
 'Bacillus thuringienses',
 'Bambusoideae sp.',
 'Banana',
 'Barley',
 'Bat',
 'Bat colony',
 'Bell pepper',
 'Bermeya goat',
 'Bhk-21',
 'Bing cherry',
 'Bird(acrocephalus schoenbaeus)',
 'Bird(andropadus virens)',
 'Bird(euplectes afra)',
 'Bird(ploceus melanocephalus)',
 'Bird(syl